# Hyper Parameters Tuning

In [ ]:
# import libraries
%pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
# import libraries
# Data handling
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Train-test split and hyperparameter tuning
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score

# Machine Learning model
from sklearn.ensemble import RandomForestRegressor

# Evaluation metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:

df=pd.read_csv("C:\\Users\\guntu\\OneDrive\\Desktop\\Hyper parameter\\nvda_stock.csv")
df

,Date,Open,High,Low,Close,Volume
0,2010-01-04,0.424242,0.426763,0.415074,0.423784,800204000
1,2010-01-05,0.422179,0.434556,0.422179,0.429972,728648000
2,2010-01-06,0.429743,0.433639,0.425617,0.432722,649168000
3,2010-01-07,0.430430,0.432264,0.421033,0.424242,547792000
4,2010-01-08,0.420804,0.428139,0.418283,0.425159,478168000
...,...,...,...,...,...,...
4103,2026-04-28,209.490005,214.729996,208.199997,213.169998,180275400
4104,2026-04-29,212.699997,212.720001,207.580002,209.250000,123711800
4105,2026-04-30,209.929993,210.300003,198.699997,199.570007,225239200
4106,2026-05-01,201.279999,203.000000,197.119995,198.449997,128647000


In [ ]:
df.head()

,Date,Open,High,Low,Close,Volume
0,2010-01-04,0.424242,0.426763,0.415074,0.423784,800204000
1,2010-01-05,0.422179,0.434556,0.422179,0.429972,728648000
2,2010-01-06,0.429743,0.433639,0.425617,0.432722,649168000
3,2010-01-07,0.430430,0.432264,0.421033,0.424242,547792000
4,2010-01-08,0.420804,0.428139,0.418283,0.425159,478168000


In [ ]:
df.shape

(4108, 6)

In [ ]:
df.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume'], dtype='str')

In [ ]:
# data preproccessing
df.isnull().sum()

Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4108 entries, 0 to 4107
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    4108 non-null   str    
 1   Open    4108 non-null   float64
 2   High    4108 non-null   float64
 3   Low     4108 non-null   float64
 4   Close   4108 non-null   float64
 5   Volume  4108 non-null   int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 192.7 KB


In [ ]:
df.describe()

,Open,High,Low,Close,Volume
count,4108.000000,4108.000000,4108.000000,4108.000000,4.108000e+03
mean,26.024894,26.451211,25.552379,26.021081,4.735287e+08
std,49.394800,50.128682,48.547739,49.364617,3.051482e+08
min,0.199859,0.207423,0.198255,0.203526,4.564400e+07
25%,0.434635,0.440246,0.428689,0.434418,2.774374e+08
50%,4.114192,4.168928,4.059048,4.104808,4.043840e+08
75%,19.244044,19.592118,18.803971,19.263086,5.809790e+08
max,212.699997,216.830002,208.199997,216.610001,3.692928e+09


In [ ]:
# drop the random rows 3108 rows from the dataset
df=df.drop(df.sample(n=3108, random_state=42).index)

In [ ]:
df.shape

(1000, 6)

In [ ]:
# handled the date column
df['Date']=pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day']=df['Date'].dt.day#drop the date column after extracting year, month, and day
df= df.drop('Date',axis=1)


In [ ]:
# divide the dataset into features and target variable
X=df.drop('Close',axis=1)
y= df['Close']

In [ ]:
# split the dataset into training and testing sets
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2,random_state=42)

In [ ]:
# scale the features using Standard Scaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled:")
print(X_train_scaled)

print("\nX_test_scaled:")
print(X_test_scaled)

X_train_scaled:
[[-0.53927421 -0.53981263 -0.53883095 ... -1.64679082  0.14003429
  -0.43666835]
 [-0.47506384 -0.47369001 -0.47459824 ...  0.24395928 -1.58789522
  -1.00450757]
 [-0.14410867 -0.13763091 -0.13905872 ...  0.66412598  1.00399904
  -1.23164325]
 ...
 [ 0.94364879  0.92416784  0.93607282 ...  1.29437601 -1.29990697
   0.01760302]
 [-0.49054882 -0.49180213 -0.48966483 ... -0.17620741 -1.01191872
   1.49398497]
 [-0.53752916 -0.5382416  -0.53722077 ... -1.43670748  1.29198729
   0.2447387 ]]

X_test_scaled:
[[-0.40266364 -0.40176306 -0.40073744 ...  0.03387594  1.00399904
  -1.68591462]
 [-0.10871931 -0.10312416 -0.13596412 ...  0.87420932 -0.72393046
   0.58544223]
 [-0.18241229 -0.18469634 -0.20064674 ...  0.87420932 -0.43594221
  -0.77737188]
 ...
 [-0.53741347 -0.53811401 -0.53699658 ... -1.01654079  0.14003429
  -1.68591462]
 [-0.29786731 -0.30221709 -0.29745269 ...  0.45404263  0.42802254
   0.35830654]
 [-0.53746931 -0.53814652 -0.5371178  ... -1.43670748  0.14003429


In [ ]:
# import model
# create an instance of RandomForestClassifier

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)
rf_model.fit(X_train_scaled, y_train)
y_pred = rf_model.predict(X_test_scaled)
# check the predictions
print("Predicted Values:")
print(y_pred)


Predicted Values:
[  7.08123905  21.25524036  17.73882723  15.56158532   1.64138432
  18.16198282  13.48518233   6.87232973  69.81907703   0.30236603
  43.04808697   0.33714076  13.25884301 177.22724442 144.29152145
 146.08316528   0.45685768 133.07060074  17.0185775    0.49011167
   4.43893753   0.27803699   0.88374466   5.28941082   0.33574553
   0.37576071   0.32757439 175.13538193 139.13847931 167.047621
   0.68415101   0.28895984  13.47578488   6.01026431   0.87044283
   0.33629772  22.01291941   5.9549325  138.7012706    0.52902963
   0.50574579   0.33633525   0.5451632  176.67952423   0.50500246
   0.45422823 116.34836967   5.95347553   2.50748795  44.24836864
   4.41686322 133.08871452   0.45725105   5.19589865   0.29033624
   0.24471278  13.07621585   0.37956811   0.37014948  44.84240704
   0.46974758   0.34176906   4.01382324   0.32865206   8.86905331
  22.14363413  15.24866055  87.25338684   0.43077869   4.41301195
   0.32694478   5.6344131    6.25492444  17.4667688    0.447

In [ ]:
# check the error
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE : 0.38465258580818795
MSE : 0.8766733691989006
RMSE: 0.9363083729193606
R2 Score: 0.9996852257305704


In [ ]:
# Check the training R2 score of the model
r2_train = r2_score(
    y_train,
    rf_model.predict(X_train_scaled)
)

# Print the training R2 score
print("Training R2 Score:", r2_train)

Training R2 Score: 0.9999442400004328


In [ ]:
r2_train = r2_score(y_train, rf_model.predict(X_train_scaled))
r2_test = r2_score(y_test, rf_model.predict(X_test_scaled))

print("Training R2 Score:", r2_train)
print("Testing R2 Score :", r2_test)

Training R2 Score: 0.9999442400004328
Testing R2 Score : 0.9996852257305704


In [ ]:
# Define the parameter grid
from sklearn.model_selection import GridSearchCV

# Define hyperparameters
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False]
}

# Create GridSearchCV
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1
)

# Train GridSearchCV
grid_search.fit(X_train_scaled, y_train)

# Best parameters
print("Best Parameters:")
print(grid_search.best_params_)

# Best CV score
print("Best CV R2 Score:")
print(grid_search.best_score_)

# Best model
best_rf_model = grid_search.best_estimator_

Best Parameters:
{'bootstrap': True, 'max_depth': 15, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best CV R2 Score:
0.9994477292539679


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

# Define hyperparameters
param_dist = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [5, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}

# Time Series Cross Validation
tscv = TimeSeriesSplit(n_splits=5)

# Randomized Search
random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_dist,
    n_iter=20,
    cv=tscv,
    scoring="r2",
    random_state=42,
    n_jobs=-1
)

# Train RandomizedSearchCV
random_search.fit(X_train_scaled, y_train)

# Best parameters
print("Best Parameters:")
print(random_search.best_params_)

# Best CV score
print("Best CV R2 Score:")
print(random_search.best_score_)

# Best model
best_rf_model = random_search.best_estimator_

Best Parameters:
{'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 5}
Best CV R2 Score:
0.9980301858987068
